# QIAS 2026 RAG Training with Qwen3 Thinking Model

**Training and Optimization Workflow for Google Colab**

This notebook provides a complete training workflow for the RAG system:
- Iterative error analysis and improvement
- Feedback loop for system optimization
- Optional model fine-tuning
- Performance tracking across iterations

## Requirements
- Google Colab with **A100 GPU** recommended
- Completed initial setup from QIAS_RAG_Colab.ipynb
- Development dataset for training

## Step 1: Setup and Initialization

In [13]:
# Verify GPU
!nvidia-smi

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

Sat Feb 14 23:01:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             51W /  400W |       6MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 2: Install Dependencies (if not done)

In [ ]:
# Install required packages
import os

# Locate requirements.txt from parent directory
req_path = os.path.abspath(os.path.join(os.getcwd(), '.', 'requirements.txt'))
print(req_path)
if os.path.exists(req_path):
    print(f"Installing dependencies from {req_path}...")
    !pip install -q -r "{req_path}"
else:
    print("requirements.txt not found, installing default dependencies...")
    # Fallback to relaxed versions
    !pip install -q "chromadb>=0.4.0" "sentence-transformers>=3.0.0"
    !pip install -q "PyMuPDF>=1.20.0" "pdfplumber>=0.10.0"
    !pip install -q "arabic-reshaper>=3.0.0" "python-bidi>=0.6.0"
    !pip install -q "rank-bm25>=0.2.0" "transformers>=4.40.0" "tokenizers<0.21"
    !pip install -q "ollama>=0.3.0" "pydantic>=2.0.0" "PyYAML>=6.0.0"
    !pip install -q pytest

print("✓ Dependencies installed")

/requirements.txt
requirements.txt not found, installing default dependencies...
✓ Dependencies installed


## Step 3: Ensure Ollama and Qwen3 are Running

In [26]:
%%bash
# Check if Ollama is running, start if not
if ! pgrep -x "ollama" > /dev/null; then
    echo "Starting Ollama..."
    nohup ollama serve > ollama.log 2>&1 &
    sleep 5
fi

# Check if model is available
ollama list | grep qwen3 || echo "⚠️ Qwen3 model not found. Run: ollama pull qwen3:30b-thinking"

Starting Ollama...
⚠️ Qwen3 model not found. Run: ollama pull qwen3:30b-thinking


bash: line 9: ollama: command not found


## Step 4: Load RAG System

In [5]:
import sys
import shutil
from pathlib import Path

# Copy RAG code from Drive if available
drive_rag_path = '/content/drive/MyDrive/QIAS_RAG/qias_rag_thinking'
if Path(drive_rag_path).exists():
    if Path('/content/qias_rag_thinking').exists():
        shutil.rmtree('/content/qias_rag_thinking')
    shutil.copytree(drive_rag_path, '/content/qias_rag_thinking')
    print("✓ Copied RAG system from Drive")
else:
    print("⚠️ RAG system not found in Drive. Please upload qias_rag_thinking/")

# Add to path
sys.path.insert(0, '/content/qias_rag_thinking')
sys.path.insert(0, '/content/qias_shared_task_2026')

# Import components
from src.rag_pipeline import RAGPipeline
from src.training.feedback_loop import FeedbackLoop
from src.evaluation.mir_e_wrapper import MIREvaluator

print("✓ Imports successful")

⚠️ RAG system not found in Drive. Please upload qias_rag_thinking/


ModuleNotFoundError: No module named 'src'

## Step 5: Initialize RAG Pipeline

In [ ]:
# Initialize pipeline
pipeline = RAGPipeline(config_path='/content/qias_rag_thinking/config/rag_config.yaml')

print("✓ RAG Pipeline initialized")

## Step 6: Build/Load Knowledge Base

In [ ]:
# Build knowledge base from PDFs
pdf_directory = '/content/drive/MyDrive/QIAS_RAG/data/pdfs'

import os
if os.path.exists(pdf_directory):
    pdf_files = [f for f in os.listdir(pdf_directory) if f.endswith('.pdf')]
    print(f"Found {len(pdf_files)} PDF files")
    
    if pdf_files:
        pipeline.build_knowledge_base(pdf_directory)
        print("\n✓ Knowledge base built")
        
        stats = pipeline.vector_store.get_collection_stats()
        print(f"Total documents: {stats['total_documents']}")
    else:
        print("⚠️ No PDFs found. Upload PDFs to continue.")
else:
    print(f"⚠️ PDF directory not found: {pdf_directory}")

## Step 7: Load Training Dataset

In [ ]:
import json

# Load QIAS dataset
dataset_file = '/content/qias_shared_task_2026/data/jsonl/mawarith/qias2025_almawarith_part1.json'

with open(dataset_file, 'r', encoding='utf-8') as f:
    full_dataset = json.load(f)

print(f"Loaded {len(full_dataset)} total cases")

# Split into train/dev/test
train_size = 20  # Adjust based on your needs
dev_size = 10
test_size = 10

train_set = full_dataset[:train_size]
dev_set = full_dataset[train_size:train_size+dev_size]
test_set = full_dataset[train_size+dev_size:train_size+dev_size+test_size]

print(f"Train: {len(train_set)}, Dev: {len(dev_set)}, Test: {len(test_set)}")

## Step 8: Run Initial Baseline Evaluation

In [ ]:
print("Running baseline evaluation on dev set...\n")

# Run on dev set
baseline_results = pipeline.batch_query(dev_set, save_results=False)

# Calculate baseline metrics
baseline_success = sum(
    1 for r in baseline_results 
    if r['parsed_output'].get('validation_success')
) / len(baseline_results)

print(f"\n{'='*80}")
print("BASELINE RESULTS")
print(f"{'='*80}")
print(f"Success Rate: {baseline_success*100:.2f}%")
print(f"Successful: {int(baseline_success * len(baseline_results))}/{len(baseline_results)}")
print(f"{'='*80}")

## Step 9: Initialize Feedback Loop

In [ ]:
# Initialize feedback loop
feedback = FeedbackLoop(config_path='/content/qias_rag_thinking/config/rag_config.yaml')

print("✓ Feedback loop initialized")

## Step 10: Analyze Baseline Errors

In [ ]:
# Analyze errors from baseline
error_analysis = feedback.analyze_errors(baseline_results, dev_set)

print("\n" + "="*80)
print("ERROR ANALYSIS")
print("="*80)
print(f"Total cases: {error_analysis['total_cases']}")
print(f"Parsing failures: {error_analysis['parsing_failures']}")
print(f"Validation failures: {error_analysis['validation_failures']}")
print(f"Low thinking quality: {error_analysis['low_thinking_quality']}")
print(f"Incorrect heirs: {len(error_analysis['incorrect_heirs'])}")
print(f"Retrieval issues: {len(error_analysis['retrieval_issues'])}")

print("\nTop Error Categories:")
for category, count in error_analysis['error_categories'].most_common(5):
    print(f"  • {category}: {count}")

## Step 11: Get Improvement Suggestions

In [ ]:
# Get suggestions for improvement
suggestions = feedback.suggest_improvements(error_analysis)

print("\n" + "="*80)
print("IMPROVEMENT SUGGESTIONS")
print("="*80)
for suggestion in suggestions:
    print(suggestion)
    print()

## Step 12: Iterative Training Loop

This will run multiple iterations, analyze errors, and track improvement.

In [ ]:
# Run iterative improvement
MAX_ITERATIONS = 3

training_history = feedback.iterative_improvement(
    pipeline=pipeline,
    dataset=train_set,
    max_iterations=MAX_ITERATIONS
)

print("\n" + "="*80)
print("TRAINING COMPLETE")
print("="*80)
print(f"Best Score: {training_history['best_score']*100:.2f}%")
print(f"Best Iteration: {training_history['best_iteration']}")
print("="*80)

## Step 13: Visualize Training Progress

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Extract metrics from history
iterations = [i['iteration'] for i in training_history['iterations']]
success_rates = [i['success_rate'] for i in training_history['iterations']]
parsing_failures = [i['error_analysis']['parsing_failures'] for i in training_history['iterations']]
validation_failures = [i['error_analysis']['validation_failures'] for i in training_history['iterations']]

# Create plots
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Success rate over iterations
axes[0].plot(iterations, [s*100 for s in success_rates], marker='o', linewidth=2, markersize=8)
axes[0].set_xlabel('Iteration', fontsize=12)
axes[0].set_ylabel('Success Rate (%)', fontsize=12)
axes[0].set_title('Training Progress: Success Rate', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 100)

# Error types over iterations
width = 0.35
x = range(len(iterations))
axes[1].bar([i - width/2 for i in x], parsing_failures, width, label='Parsing Failures', alpha=0.8)
axes[1].bar([i + width/2 for i in x], validation_failures, width, label='Validation Failures', alpha=0.8)
axes[1].set_xlabel('Iteration', fontsize=12)
axes[1].set_ylabel('Number of Errors', fontsize=12)
axes[1].set_title('Error Types by Iteration', fontsize=14, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(iterations)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Save to Drive
plt.savefig('/content/drive/MyDrive/QIAS_RAG/results/training_progress.png', dpi=300, bbox_inches='tight')
print("\n✓ Training progress saved to Drive")

## Step 14: Final Evaluation on Test Set

In [ ]:
print("Running final evaluation on test set...\n")

# Run on test set
test_results = pipeline.batch_query(test_set, save_results=True)

# Calculate test metrics
test_success = sum(
    1 for r in test_results 
    if r['parsed_output'].get('validation_success')
) / len(test_results)

# Average thinking quality
avg_thinking = sum(
    r.get('thinking_quality', {}).get('quality_score', 0)
    for r in test_results
) / len(test_results)

print(f"\n{'='*80}")
print("FINAL TEST RESULTS")
print(f"{'='*80}")
print(f"Success Rate: {test_success*100:.2f}%")
print(f"Successful: {int(test_success * len(test_results))}/{len(test_results)}")
print(f"Avg Thinking Quality: {avg_thinking:.2f}")
print(f"\nImprovement from baseline: {(test_success - baseline_success)*100:+.2f}%")
print(f"{'='*80}")

## Step 15: Run MIR-E Evaluation (if available)

In [ ]:
# Run MIR-E evaluation if evaluation module is available
try:
    evaluator = MIREvaluator()
    
    # Evaluate
    predictions_file = '/content/qias_rag_thinking/results/rag_predictions.json'
    
    if Path(predictions_file).exists():
        mir_results = evaluator.evaluate_predictions(predictions_file, dataset_file)
        
        print(f"\n{'='*80}")
        print("MIR-E EVALUATION")
        print(f"{'='*80}")
        print(f"MIR-E Score: {mir_results.get('average_mir_e', 0):.4f}")
        print(f"\nSubscores:")
        for key, value in mir_results.get('subscores', {}).items():
            print(f"  {key}: {value:.4f}")
        print(f"{'='*80}")
    else:
        print("⚠️ Predictions file not found for MIR-E evaluation")
        
except Exception as e:
    print(f"⚠️ MIR-E evaluation not available: {e}")

## Step 16: Save Training Results to Drive

In [ ]:
# Save training history
results_dir = Path('/content/drive/MyDrive/QIAS_RAG/results')
results_dir.mkdir(parents=True, exist_ok=True)

# Save history as JSON
with open(results_dir / 'training_history.json', 'w', encoding='utf-8') as f:
    json.dump(training_history, f, ensure_ascii=False, indent=2)

# Save summary
summary = {
    'baseline_success_rate': baseline_success,
    'final_test_success_rate': test_success,
    'improvement': test_success - baseline_success,
    'best_training_iteration': training_history['best_iteration'],
    'best_training_score': training_history['best_score'],
    'avg_thinking_quality': avg_thinking,
    'num_train_cases': len(train_set),
    'num_test_cases': len(test_set)
}

with open(results_dir / 'training_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

# Copy all error analysis files
!cp -r /content/qias_rag_thinking/results/* /content/drive/MyDrive/QIAS_RAG/results/

print("\n✓ All results saved to Google Drive")
print(f"Location: {results_dir}")
print("\nFiles saved:")
!ls -lh /content/drive/MyDrive/QIAS_RAG/results/

## Step 17: Training Summary Report

In [ ]:
# Display comprehensive summary
print("\n" + "="*80)
print("TRAINING SUMMARY REPORT")
print("="*80)
print(f"\n📊 Dataset Distribution:")
print(f"   Training: {len(train_set)} cases")
print(f"   Dev: {len(dev_set)} cases")
print(f"   Test: {len(test_set)} cases")

print(f"\n📈 Performance Metrics:")
print(f"   Baseline Success Rate: {baseline_success*100:.2f}%")
print(f"   Final Test Success Rate: {test_success*100:.2f}%")
print(f"   Improvement: {(test_success - baseline_success)*100:+.2f}%")
print(f"   Avg Thinking Quality: {avg_thinking:.2f}/1.0")

print(f"\n🔄 Training Progress:")
print(f"   Total Iterations: {MAX_ITERATIONS}")
print(f"   Best Training Score: {training_history['best_score']*100:.2f}%")
print(f"   Best Iteration: {training_history['best_iteration']}")

if test_success >= 0.99:
    print("\n🎉 " + "="*76 + " 🎉")
    print("   ✓ TARGET ACHIEVED: >99% SUCCESS RATE!")
    print("   🎉 " + "="*76 + " 🎉")
elif test_success >= 0.95:
    print("\n⭐ Great progress! Close to target. Consider:")
    print("     • Adding more high-quality PDFs")
     • Adjusting retrieval parameters")
    print("     • Fine-tuning the model (optional)")
else:
    print("\n📋 Next Steps to Improve:")
    print("     1. Review error_analysis_iter_*.json files")
    print("     2. Add more comprehensive Islamic law PDFs")
    print("     3. Increase training dataset size")
    print("     4. Adjust retrieval and reranking parameters")
    print("     5. Consider model fine-tuning if needed")

print("\n" + "="*80)

## Optional: Model Fine-tuning

If you need to fine-tune the Qwen3 model for better performance, run the cells below.

In [ ]:
# Install fine-tuning dependencies
!pip install -q unsloth trl

print("✓ Fine-tuning dependencies installed")

In [ ]:
from src.training.fine_tune import FineTuner

# Initialize fine-tuner
tuner = FineTuner()

# Prepare training data
tuner.prepare_training_data(
    dataset=full_dataset[:100],  # Use more data for actual fine-tuning
    output_file='/content/training_data.jsonl'
)

# Generate training script
tuner.save_training_script('/content/train_qwen3.py')

print("\n✓ Fine-tuning data prepared")
print("\nTo train:")
print("  !python /content/train_qwen3.py")

## Troubleshooting

**Low Success Rate:**
- Check if PDFs are properly loaded
- Verify Ollama and Qwen3 are running
- Review error_analysis files for specific issues

**Out of Memory:**
- Reduce batch size in config
- Use smaller model (qwen3:7b-thinking)
- Clear GPU memory: `torch.cuda.empty_cache()`

**Parsing Failures:**
- Check prompt formatting
- Add more few-shot examples
- Adjust temperature (try 0.1 instead of 0.0)

## Next Steps

1. ✅ Review training results in Google Drive
2. ✅ Apply improvements based on error analysis
3. ✅ Re-run training with adjustments
4. ✅ Once >99% achieved, run full evaluation
5. ✅ Deploy to production